# Create or append Virtual Icechunk Store from SHYFEM forecast NetCDF files

* This notebook appends Taranto SHYFEM forecast data to Icechunk store using date-based set logic.
* If no repo exists, a new one will be created with references to all the existing NetCDF files. 

**Append Methodology:**
1. **`set_repo`**: extract all dates currently present in the Icechunk store's `time` coordinate
2. **`set_cloud`**: Scan the S3 bucket for all available NOS files and extract their dates.
3. **`new_dates`**: Calculate the difference (`set_cloud - set_repo`) to determine exactly which days need to be written for creation or appended.


In [1]:
import warnings
import os
import pandas as pd
import fsspec
import xarray as xr
from pathlib import Path
#from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import S3Store

In [3]:
os.environ['AWS_REQUEST_CHECKSUM_CALCULATION']='when_required'
os.environ['AWS_RESPONSE_CHECKSUM_VALIDATION']='when_required'

In [4]:
# Load credentials
#_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env', override=True)

# Configuration
storage_endpoint = "https://s3.kopah.uw.edu"
storage_bucket = "liveocean-share"
storage_name = 'icechunk-test-1'
bucket_url = f"s3://{storage_bucket}"

# Setup Filesystem
fs = fsspec.filesystem('s3', anon=False, endpoint_url=storage_endpoint, 
                       skip_instance_cache=True, use_listings_cache=False)

In [5]:
os.environ

environ{'TERM_PROGRAM': 'Apple_Terminal',
        'PROJ_DATA': '/Users/parkermaccready/miniconda3/envs/loenv/share/proj',
        'SHELL': '/bin/bash',
        'TERM': 'xterm-color',
        'HOMEBREW_REPOSITORY': '/opt/homebrew',
        'TMPDIR': '/var/folders/31/rhr_qv2x2v1bjlwbnmm89lth0000gn/T/',
        'CONDA_SHLVL': '2',
        'CONDA_PROMPT_MODIFIER': '(loenv) ',
        'TERM_PROGRAM_VERSION': '453',
        'GSETTINGS_SCHEMA_DIR_CONDA_BACKUP': '',
        'OLDPWD': '/Users/parkermaccready/Documents/LO',
        'TERM_SESSION_ID': '1AB5207A-12EB-4C42-A6DA-47713EA6501A',
        'USER': 'parkermaccready',
        'UDUNITS2_XML_PATH': '/Users/parkermaccready/miniconda3/envs/loenv/share/udunits/udunits2.xml',
        'CONDA_EXE': '/Users/parkermaccready/miniconda3/bin/conda',
        'SSH_AUTH_SOCK': '/private/tmp/com.apple.launchd.gUiYEbRCjD/Listeners',
        'BASH_SILENCE_DEPRECATION_WARNING': '1',
        '_CE_CONDA': '',
        'CONDA_PREFIX_1': '/Users/parkermaccready/mi

In [6]:
fs.upload('./CLAUDE.md','s3://liveocean-share/Claude.md')

[None]

In [7]:
os.environ

environ{'TERM_PROGRAM': 'Apple_Terminal',
        'PROJ_DATA': '/Users/parkermaccready/miniconda3/envs/loenv/share/proj',
        'SHELL': '/bin/bash',
        'TERM': 'xterm-color',
        'HOMEBREW_REPOSITORY': '/opt/homebrew',
        'TMPDIR': '/var/folders/31/rhr_qv2x2v1bjlwbnmm89lth0000gn/T/',
        'CONDA_SHLVL': '2',
        'CONDA_PROMPT_MODIFIER': '(loenv) ',
        'TERM_PROGRAM_VERSION': '453',
        'GSETTINGS_SCHEMA_DIR_CONDA_BACKUP': '',
        'OLDPWD': '/Users/parkermaccready/Documents/LO',
        'TERM_SESSION_ID': '1AB5207A-12EB-4C42-A6DA-47713EA6501A',
        'USER': 'parkermaccready',
        'UDUNITS2_XML_PATH': '/Users/parkermaccready/miniconda3/envs/loenv/share/udunits/udunits2.xml',
        'CONDA_EXE': '/Users/parkermaccready/miniconda3/bin/conda',
        'SSH_AUTH_SOCK': '/private/tmp/com.apple.launchd.gUiYEbRCjD/Listeners',
        'BASH_SILENCE_DEPRECATION_WARNING': '1',
        '_CE_CONDA': '',
        'CONDA_PREFIX_1': '/Users/parkermaccready/mi

In [9]:
#fs.ls(f'{storage_bucket}/icechunk')

In [10]:
fs.rm('liveocean-share/icechunk/icechunk-test-1',recursive=True)

FileNotFoundError: ['liveocean-share/icechunk/icechunk-test-1']

In [11]:
# Define Icechunk Storage & Config
storage = icechunk.s3_storage(
    bucket=storage_bucket,
    prefix=f"icechunk/{storage_name}",
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',
    force_path_style=True,
)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"{bucket_url}/",
        store=icechunk.s3_store(region="not-used", anonymous=False, s3_compatible=True, 
                                force_path_style=True, endpoint_url=storage_endpoint),
    ),
)

credentials = icechunk.containers_credentials({f"{bucket_url}/": icechunk.s3_credentials(anonymous=False)})

store_obj = S3Store(
    bucket=storage_bucket,
    endpoint=storage_endpoint, # e.g., "https://rustfs.vm.fedcloud.eu:9001"
    region="not-used",
)

registry = ObjectStoreRegistry({bucket_url: store_obj})
parser = HDFParser()

### Step 1: Create Date Sets (`set_repo` vs `set_cloud`)

In [12]:
# --- 1. Get Dates from Icechunk Repo (set_repo) ---
try:
    repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)
    session = repo.readonly_session("main")
    ds = xr.open_zarr(session.store, consolidated=False, chunks={})
    
    if 'ocean_time' in ds.coords:
        # Extract dates as YYYYMMDD strings
        dates = pd.to_datetime(ds.time.values) + pd.Timedelta(days=1)
        set_repo = set(dates.strftime('%Y%m%d'))
    else:
        set_repo = set()
        
except Exception as e:
    print(f"Repo access failed or empty ({e}). Assuming set_repo is empty.")
    repo = None
    set_repo = set()

print(f"set_repo: {len(set_repo)} dates found.")

# s3://liveocean-share/f2026.05.08/layers.nc

# --- 2. Get Dates from Cloud Bucket (set_cloud) ---
print("Scanning S3 for liveocean files...")
nos_files = fs.glob(f'{bucket_url}/*/layers.nc')

set_cloud = set()
date_to_files_map = {} # Helper to quickly find files for a date later

for f in nos_files:
    # Path structure: .../forecast/YYYYMMDD/taranto_nos_YYYYMMDD_nc4.nc
    try:        
        date_str0 = f.split('/')[-2] # Parent folder is date
        date_str = date_str0.replace('.','').replace('f','')
        set_cloud.add(date_str)
        
        # Store both NOS and assume OUS follows same pattern
        base_dir = os.path.dirname(f)
        nos_path = f's3://{f}'
        #ous_path = f's3://{base_dir}/taranto_ous_{date_str}_nc4.nc'
        
        date_to_files_map[date_str] = {'nos': nos_path}#, 'ous': ous_path}
        
    except IndexError:
        pass

print(f"set_cloud: {len(set_cloud)} dates found.")

Repo access failed or empty (  x the repository doesn't exist
  | 
  | context:
  |    0: icechunk::repository::open
  |              at icechunk/src/repository.rs:343
  | 
  `-> the repository doesn't exist
). Assuming set_repo is empty.
set_repo: 0 dates found.
Scanning S3 for liveocean files...
set_cloud: 318 dates found.


In [13]:
# --- 3. Determine New Dates ---
new_dates = sorted(list(set_cloud - set_repo))

print(f"Dates to process: {len(new_dates)}")
if new_dates:
    print(f"Range: {new_dates[0]} to {new_dates[-1]}")

Dates to process: 318
Range: 20250715 to 20260529


In [14]:
nos_files[-1]

'liveocean-share/f2026.05.29/layers.nc'

### Step 2: Virtualize and Merge New Data

In [15]:
# def fix_ds(ds):
#     """Standardizes dimensions and coordinates for Taranto dataset."""
#     ds = ds.rename_vars(time='valid_time')
#     ds = ds.rename_dims(time='step')
#     step = (ds.valid_time - ds.valid_time[0]).assign_attrs({"standard_name": "forecast_period"})
#     time = ds.valid_time[0].assign_attrs({"standard_name": "forecast_reference_time"})
#     ds = ds.assign_coords(step=step, time=time)
#     ds = ds.drop_indexes("valid_time")
#     ds = ds.drop_vars('valid_time')
#     ds = ds.set_coords(['latitude', 'longitude', 'element_index', 'topology', 'total_depth'])
#     return ds

def fix_ds(ds):
    subset_ds = ds.isel(ocean_time=slice(0, 6))
    return subset_ds

In [16]:
nos_urls = [date_to_files_map[d]['nos'] for d in new_dates]
nos_urls[0]

's3://liveocean-share/f2025.07.15/layers.nc'

In [17]:
ds_final = None

new_dates = new_dates[:3]  # only if testing

if new_dates:
    # Reconstruct file lists based on the identified dates
    nos_urls = [date_to_files_map[d]['nos'] for d in new_dates]
    
    # --- Process NOS ---
    print(f"Virtualizing {len(nos_urls)} NOS files...")
    nos_list = [
        open_virtual_dataset(url, parser=parser, registry=registry, loadable_variables=["ocean_time"])
        for url in nos_urls
    ]
    nos_list = [fix_ds(ds) for ds in nos_list]
    combined_nos = xr.concat(
        nos_list, dim="ocean_time", coords="minimal", compat="override", combine_attrs="override"
    )
    
else:
    print("No new dates found. Skipping processing.")

Virtualizing 3 NOS files...


/var/folders/31/rhr_qv2x2v1bjlwbnmm89lth0000gn/T/ipykernel_55224/2557519899.py:16: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  combined_nos = xr.concat(


In [18]:
combined_nos

<xarray.Dataset> Size: 5GB
Dimensions:                (ocean_time: 18, eta_rho: 1302, xi_rho: 663)
Coordinates:
  * ocean_time             (ocean_time) datetime64[ns] 144B 2025-07-15 ... 20...
Dimensions without coordinates: eta_rho, xi_rho
Data variables: (12/46)
    lon_rho                (ocean_time, eta_rho, xi_rho) float64 124MB Manife...
    lat_rho                (ocean_time, eta_rho, xi_rho) float64 124MB Manife...
    mask_rho               (ocean_time, eta_rho, xi_rho) float64 124MB Manife...
    h                      (ocean_time, eta_rho, xi_rho) float64 124MB Manife...
    temp_surface           (ocean_time, eta_rho, xi_rho) float32 62MB Manifes...
    salt_surface           (ocean_time, eta_rho, xi_rho) float32 62MB Manifes...
    ...                     ...
    salt_bottom            (ocean_time, eta_rho, xi_rho) float32 62MB Manifes...
    phytoplankton_bottom   (ocean_time, eta_rho, xi_rho) float32 62MB Manifes...
    NO3_bottom             (ocean_time, eta_rho, xi_rho) float32 62MB Manifes...
    oxygen_bottom          (ocean_time, eta_rho, xi_rho) float32 62MB Manifes...
    PH_bottom              (ocean_time, eta_rho, xi_rho) float64 124MB Manife...
    ARAG_bottom            (ocean_time, eta_rho, xi_rho) float64 124MB Manife...
Attributes:
    history:                Tue Jul 15 14:38:56 2025: ncrcat -p /dat1/parker/...
    NCO:                    netCDF Operators version 5.2.7 (Homepage = http:/...
    nco_input_file_number:  19
    nco_input_file_list:    layers_000000.nc layers_000001.nc layers_000002.n...

### Step 3: Append to Icechunk

In [19]:
storage

<icechunk.Storage>
type: S3 (native)
bucket: liveocean-share
prefix: icechunk/icechunk-test-1
region: not-used
endpoint_url: https://s3.kopah.uw.edu
force_path_style: True

In [20]:
ds_final = combined_nos

if ds_final is not None:
    # Ensure we have a valid repo object
    if repo is None:
        repo = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=credentials)
        initial_session = repo.writable_session("main")

        # Append
        print(f"Writing {len(ds_final.ocean_time)} time steps to Icechunk...")
        ds_final.virtualize.to_icechunk(initial_session.store)
    
        # Commit
        msg = f"Initialized with forecast data: {new_dates[0]} to {new_dates[-1]}"
        initial_session.commit(msg)
        print(f"Commit successful: '{msg}'")
    # Create Writable Session
    else:
        append_session = repo.writable_session("main")

        # Append
        print(f"Appending {len(ds_final.ocean_time)} time steps to Icechunk...")
        ds_final.virtualize.to_icechunk(append_session.store, append_dim="ocean_time")
    
        # Commit
        msg = f"Appended forecast data: {new_dates[0]} to {new_dates[-1]}"
        append_session.commit(msg)
        print(f"Commit successful: '{msg}'")

    # Verify History
    history = repo.ancestry(branch="main")
    latest = next(history)
    print(f"Latest Commit [{latest.written_at}]: {latest.message}")
    
else:
    print("Nothing to append.")

Writing 18 time steps to Icechunk...
Commit successful: 'Initialized with forecast data: 20250715 to 20250717'
Latest Commit [2026-05-29 19:07:12.953627+00:00]: Initialized with forecast data: 20250715 to 20250717
